# 머신러닝 실습 : 고객 구매 데이터로 성별 예측 모델링 (분류 문제)
주어진 데이터는 백화점 고객의 1년 간 구매 데이터입니다. 
고객 3,500명에 대한 학습용 데이터(y.csv, X.csv)를 이용하여 성별예측 모형을 만들어보세요. 
모델의 성능은 자유롭게 측정해봅니다! 

## 0. 라이브러리 불러오기

In [80]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings(action='ignore')

## 1. 데이터 불러오기

In [157]:
X = pd.read_csv('./data/X.csv', encoding='euc-kr')
y = pd.read_csv('./data/y.csv', encoding='euc-kr')

In [158]:

X.head()

,cust_id,총구매액,최대구매액,환불금액,주구매상품,주구매지점,내점일수,내점당구매건수,주말방문비율,구매주기
0,0,68282840,11264000,6860000.0,기타,강남점,19,3.894737,0.527027,17
1,1,2136000,2136000,300000.0,스포츠,잠실점,2,1.500000,0.000000,1
2,2,3197000,1639000,NaN,남성 캐주얼,관악점,2,2.000000,0.000000,1
3,3,16077620,4935000,NaN,기타,광주점,18,2.444444,0.318182,16
4,4,29050000,24000000,NaN,보석,본 점,2,1.500000,0.000000,85


In [159]:

y.head()

,cust_id,gender
0,0,0
1,1,0
2,2,1
3,3,1
4,4,0


In [160]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   cust_id  3500 non-null   int64  
 1   총구매액     3500 non-null   int64  
 2   최대구매액    3500 non-null   int64  
 3   환불금액     1205 non-null   float64
 4   주구매상품    3500 non-null   str    
 5   주구매지점    3500 non-null   str    
 6   내점일수     3500 non-null   int64  
 7   내점당구매건수  3500 non-null   float64
 8   주말방문비율   3500 non-null   float64
 9   구매주기     3500 non-null   int64  
dtypes: float64(3), int64(5), str(2)
memory usage: 273.6 KB


In [161]:
print("X의 cust_id 중복 개수:", X['cust_id'].duplicated().sum())
print("y의 cust_id 중복 개수:", y['cust_id'].duplicated().sum())

X의 cust_id 중복 개수: 0
y의 cust_id 중복 개수: 0


In [162]:
y['gender'].value_counts() #0 여성, 1 남성

gender
0    2184
1    1316
Name: count, dtype: int64

In [163]:
y["gender"].value_counts(normalize=True, dropna=False).sort_index() * 100

gender
0    62.4
1    37.6
Name: proportion, dtype: float64

해석
- 완전한 균형은 아니지만 심한 불균형도 아님
- 6:4 비율 수준 → 일반적인 이진분류 문제에서 허용 가능한 수준
- 현재 단계에서는 class_weight 없이 기본 모델로 먼저 학습 진행
- 이후 성능이 한쪽 클래스에 치우치면 class_weight='balanced' 적용 고려

In [164]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   cust_id  3500 non-null   int64  
 1   총구매액     3500 non-null   int64  
 2   최대구매액    3500 non-null   int64  
 3   환불금액     1205 non-null   float64
 4   주구매상품    3500 non-null   str    
 5   주구매지점    3500 non-null   str    
 6   내점일수     3500 non-null   int64  
 7   내점당구매건수  3500 non-null   float64
 8   주말방문비율   3500 non-null   float64
 9   구매주기     3500 non-null   int64  
dtypes: float64(3), int64(5), str(2)
memory usage: 273.6 KB


In [165]:
X.describe()

,cust_id,총구매액,최대구매액,환불금액,내점일수,내점당구매건수,주말방문비율,구매주기
count,3500.000000,3.500000e+03,3.500000e+03,1.205000e+03,3500.000000,3500.000000,3500.000000,3500.000000
mean,1749.500000,9.191925e+07,1.966424e+07,2.407822e+07,19.253714,2.834963,0.307246,20.958286
std,1010.507298,1.635065e+08,3.199235e+07,4.746453e+07,27.174942,1.912368,0.289752,24.748682
min,0.000000,-5.242152e+07,-2.992000e+06,5.600000e+03,1.000000,1.000000,0.000000,0.000000
25%,874.750000,4.747050e+06,2.875000e+06,2.259000e+06,2.000000,1.666667,0.027291,4.000000
50%,1749.500000,2.822270e+07,9.837000e+06,7.392000e+06,8.000000,2.333333,0.256410,13.000000
75%,2624.250000,1.065079e+08,2.296250e+07,2.412000e+07,25.000000,3.375000,0.448980,28.000000
max,3499.000000,2.323180e+09,7.066290e+08,5.637530e+08,285.000000,22.083333,1.000000,166.000000


In [166]:
# X에서 cust_id 제거
X = X.drop(columns=["cust_id"])

# y에서 cust_id 제거(타깃만 남김)
y = y["gender"]

In [168]:
missing_count = X.isna().sum().sort_values(ascending=False)
missing_count

총구매액       0
최대구매액      0
환불금액       0
주구매상품      0
주구매지점      0
내점일수       0
내점당구매건수    0
주말방문비율     0
구매주기       0
dtype: int64

In [167]:
X = X.fillna(0)

In [169]:
cat_cols = X.select_dtypes(include=['object'])
col_list = cat_cols.columns.to_list() #chaining
col_list

['주구매상품', '주구매지점']

In [170]:
for col in col_list:
  # print(col)
  print(X[col].nunique())
  print(X[col].value_counts().head()) #범주형변수의 값의 종류 

42
주구매상품
기타      595
가공식품    546
농산물     339
화장품     264
시티웨어    213
Name: count, dtype: int64
24
주구매지점
본  점    1077
잠실점      474
분당점      436
부산본점     245
영등포점     241
Name: count, dtype: int64


In [171]:
# 필요한 라이브러리 불러오기
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [172]:
# 범주형 변수 인코딩 (LabelEncoder 사용)
le = LabelEncoder()

for col in cat_cols:
    X[col] = le.fit_transform(X[col])
    print(f"{col} 인코딩 완료")

X.head()

주구매상품 인코딩 완료
주구매지점 인코딩 완료


,총구매액,최대구매액,환불금액,주구매상품,주구매지점,내점일수,내점당구매건수,주말방문비율,구매주기
0,68282840,11264000,6860000.0,5,0,19,3.894737,0.527027,17
1,2136000,2136000,300000.0,21,19,2,1.500000,0.000000,1
2,3197000,1639000,0.0,6,1,2,2.000000,0.000000,1
3,16077620,4935000,0.0,5,2,18,2.444444,0.318182,16
4,29050000,24000000,0.0,15,8,2,1.500000,0.000000,85


In [173]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [174]:
scaler = StandardScaler() #클래스 > import
X_scaled = scaler.fit_transform(X)
# X_scaled[:3]
X_scaled #numpy의 array

array([[-0.14458009, -0.26260786, -0.04750476, ...,  0.55424743,
         0.75862274, -0.15996211],
       [-0.54918957, -0.54796686, -0.26546133, ..., -0.69816782,
        -1.06053002, -0.80655356],
       [-0.5426996 , -0.56350404, -0.27542885, ..., -0.43667453,
        -1.06053002, -0.80655356],
       ...,
       [-0.56179637, -0.61239772, -0.27542885, ..., -0.95966112,
        -1.06053002, -0.84696552],
       [-0.55078606, -0.58348042, -0.27542885, ..., -0.95966112,
        -1.06053002,  0.72910114],
       [ 1.04709431,  0.46792117, -0.07697541, ..., -0.21646965,
         0.55277658, -0.5236698 ]], shape=(3500, 9))

In [175]:

# 학습/테스트 데이터 분리 (8:2 비율)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y #y가 불균형이므로 설정
)
X_train.shape, X_test.shape

((2800, 9), (700, 9))

In [176]:
from sklearn.linear_model import LogisticRegression

In [177]:
X_train, X_test, y_train, y_test

(array([[ 0.08226136,  0.16680616, -0.27542885, ..., -0.54127184,
          1.70085057, -0.32160997],
        [-0.553924  , -0.58191732, -0.27542885, ..., -0.95966112,
          0.66533285,  1.77981225],
        [-0.49402456, -0.36073909, -0.27542885, ..., -0.43667453,
          0.23386713,  1.49692849],
        ...,
        [-0.5511921 , -0.58660662, -0.27542885, ...,  1.39377854,
          1.13602273,  2.79011139],
        [ 0.67521934, -0.2658591 , -0.26778708, ...,  1.8232033 ,
          0.05664682, -0.72572963],
        [ 4.5535857 ,  1.60485543,  0.44223302, ...,  0.04344431,
          0.21932335, -0.6449057 ]], shape=(2800, 9)),
 array([[-0.55522077, -0.57879113, -0.27542885, ..., -0.95966112,
          2.39119572, -0.84696552],
        [-0.52677136, -0.46624814, -0.21402889, ..., -0.78533225,
          1.52826429, -0.60449373],
        [-0.53922524, -0.54665386, -0.27542885, ..., -0.69816782,
         -1.06053002, -0.36202194],
        ...,
        [ 0.09294307,  0.26609408,  0

In [178]:
lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")

In [179]:
lr_model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [180]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [181]:
# 예측
lr_pred = lr_model.predict(X_test)

In [182]:
def evaluate_classifier(y_true, y_pred):
    # 정확도: 전체 정답 비율
    acc = accuracy_score(y_true, y_pred)

    # 정밀도/재현율/F1: 기본적으로 positive label = 1 기준
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    # 혼동행렬: [[TN, FP], [FN, TP]]
    cm = confusion_matrix(y_true, y_pred)

    print("Accuracy :", round(acc, 4))
    print("Precision:", round(prec, 4))
    print("Recall   :", round(rec, 4))
    print("F1       :", round(f1, 4))
    print("\nConfusion Matrix")
    print(cm)

In [183]:
evaluate_classifier(y_test, lr_pred)

Accuracy : 0.55
Precision: 0.4333
Recall   : 0.6426
F1       : 0.5176

Confusion Matrix
[[216 221]
 [ 94 169]]


In [184]:
from sklearn.tree import DecisionTreeClassifier
dt_model=DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_pred=dt_model.predict(X_test)
evaluate_classifier(y_test, dt_pred)

Accuracy : 0.5657
Precision: 0.4232
Recall   : 0.4297
F1       : 0.4264

Confusion Matrix
[[283 154]
 [150 113]]


In [185]:
dt_model

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current

In [186]:
from sklearn.ensemble import RandomForestClassifier
rf_model=RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X_train, y_train)
rf_pred=rf_model.predict(X_test)
evaluate_classifier(y_test, rf_pred)

Accuracy : 0.6286
Precision: 0.5087
Recall   : 0.3346
F1       : 0.4037

Confusion Matrix
[[352  85]
 [175  88]]


In [189]:
rf_model_1=RandomForestClassifier(
  random_state=42,
  n_estimators=300,
  max_depth=10,
  min_samples_leaf=5,
  class_weight="balanced"
)
rf_model_1.fit(X_train, y_train)
rf_pred_1=rf_model_1.predict(X_test)
evaluate_classifier(y_test, rf_pred_1)

Accuracy : 0.61
Precision: 0.4828
Recall   : 0.5323
F1       : 0.5063

Confusion Matrix
[[287 150]
 [123 140]]


In [190]:
from xgboost import XGBClassifier
xgb_model=XGBClassifier(
  random_state=42,
  n_estimators=100,
  eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
xgb_pred=xgb_model.predict(X_test)
evaluate_classifier(y_test, xgb_pred)

Accuracy : 0.6371
Precision: 0.5207
Recall   : 0.4297
F1       : 0.4708

Confusion Matrix
[[333 104]
 [150 113]]


In [194]:
xgb_w=XGBClassifier(
  random_state=42,
  n_estimators=300,
  learn_rate=0.05,
  max_depth=4,
  subsample=0.8,
  colsample_bytree=0.8,
  eval_metric='logloss',
  scale_pos_weight=1.66
)
xgb_w.fit(X_train, y_train)
pred_xgb_w=xgb_w.predict(X_test)
evaluate_classifier(y_test, pred_xgb_w)

Accuracy : 0.6214
Precision: 0.4962
Recall   : 0.4981
F1       : 0.4972

Confusion Matrix
[[304 133]
 [132 131]]


In [195]:
proba = xgb_w.predict_proba(X_test)[:, 1]
proba

array([7.10092425e-01, 3.20365280e-01, 3.64352167e-01, 9.96106088e-01,
       1.78763703e-01, 4.54514056e-01, 5.39461374e-02, 8.09052214e-02,
       9.75403428e-01, 2.60185272e-01, 8.99281632e-03, 8.73750269e-01,
       7.34530985e-01, 1.59676224e-01, 9.48481977e-01, 4.98013198e-01,
       8.57741475e-01, 5.02201021e-01, 2.09305763e-01, 6.11862242e-01,
       7.82562494e-01, 7.12513506e-01, 3.64646971e-01, 4.68580663e-01,
       4.24549490e-01, 1.45637468e-02, 6.19229674e-01, 1.53841987e-01,
       9.50372577e-01, 6.07357740e-01, 3.94206315e-01, 2.15962976e-02,
       9.96474683e-01, 1.83320977e-02, 4.61601794e-01, 9.85186040e-01,
       3.24730903e-01, 7.98710704e-01, 1.81106374e-01, 7.03984797e-01,
       4.24616069e-01, 3.60787511e-02, 5.09446952e-03, 6.64911449e-01,
       8.35388675e-02, 5.96825108e-02, 2.13602334e-01, 2.80473214e-02,
       1.04310989e-01, 7.16624677e-01, 4.81357247e-01, 1.56125024e-01,
       8.83009136e-01, 2.51135111e-01, 9.05402660e-01, 6.04649305e-01,
      

In [196]:
rows = []
for thr in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]: #임계값에 따라 0/1 인지 판정만 다시한다.
    pred_thr = (proba >= thr).astype(int)

    rows.append({
        "threshold": thr,
        "accuracy": round(accuracy_score(y_test, pred_thr), 4),
        "precision": round(precision_score(y_test, pred_thr), 4),
        "recall": round(recall_score(y_test, pred_thr), 4),
        "f1": round(f1_score(y_test, pred_thr), 4)
    })

thr_result = pd.DataFrame(rows)
display(thr_result)

best_thr = float(thr_result.sort_values("f1", ascending=False).iloc[0]["threshold"])
best_pred = (proba >= best_thr).astype(int)

print("F1 기준 best threshold:", best_thr)
print("Confusion Matrix\n", confusion_matrix(y_test, best_pred))

,threshold,accuracy,precision,recall,f1
0,0.35,0.5914,0.4669,0.6160,0.5311
1,0.40,0.6029,0.4769,0.5894,0.5272
2,0.45,0.6043,0.4762,0.5323,0.5027
3,0.50,0.6214,0.4962,0.4981,0.4972
4,0.55,0.6243,0.5000,0.4639,0.4813
5,0.60,0.6271,0.5048,0.4030,0.4482


F1 기준 best threshold: 0.35
Confusion Matrix
 [[252 185]
 [101 162]]


In [197]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

In [198]:
# 1. 테스트해볼 파라미터들을 딕셔너리 형태로 정합니다.
param_grid = {
    'n_estimators': [100, 200],       # 나무의 개수
    'max_depth': [None, 10, 20],      # 나무의 최대 깊이 (너무 깊으면 과적합 위험)
    'min_samples_split': [2, 5, 10],  # 노드를 나누기 위한 최소 샘플 수
    'min_samples_leaf': [1, 2, 4]     # 리프 노드가 되기 위한 최소 샘플 수
}

# 2. 랜덤 포레스트 모델 생성
rf = RandomForestClassifier(random_state=42)

# 3. GridSearchCV 설정
# scoring='f1'을 써서 F1-Score가 가장 높은 조합을 찾도록 합니다.
# cv=5는 데이터를 5등분해서 교차 검증을 하겠다는 뜻입니다.
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, 
                           cv=5, scoring='f1', n_jobs=-1)

# 4. 학습 시작 (컴퓨터가 열심히 조합을 맞춰봅니다)
grid_search.fit(X_train, y_train)

# 5. 최적의 파라미터와 최고 점수 확인
print(f"최적 파라미터: {grid_search.best_params_}")
print(f"최고 F1 Score: {grid_search.best_score_:.4f}")

# 6. 최적의 모델로 예측하기
best_rf = grid_search.best_estimator_
rf_pred = best_rf.predict(X_test)

# 평가 함수 호출 (사용하시던 evaluate_classifier 함수)
evaluate_classifier(y_test, rf_pred)

최적 파라미터: {'max_depth': 20, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 200}
최고 F1 Score: 0.4295
Accuracy : 0.64
Precision: 0.5333
Recall   : 0.3346
F1       : 0.4112

Confusion Matrix
[[360  77]
 [175  88]]


In [199]:
from sklearn.ensemble import RandomForestClassifier

# 1. GridSearchCV로 찾은 최적 파라미터에 class_weight만 추가합니다.
rf_balanced = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=4,
    min_samples_split=2,
    random_state=42,
    class_weight='balanced'  # <--- 핵심! 부족한 클래스에 가중치를 자동으로 부여
)

# 2. 모델 학습
rf_balanced.fit(X_train, y_train)

# 3. 예측
rf_pred_balanced = rf_balanced.predict(X_test)

# 4. 결과 확인 (사용하시던 evaluate_classifier 함수)
evaluate_classifier(y_test, rf_pred_balanced)

Accuracy : 0.6357
Precision: 0.5168
Recall   : 0.4677
F1       : 0.491

Confusion Matrix
[[322 115]
 [140 123]]
